In [1]:
"""
Conda setup:
conda create -n smap python=3.11 -y
conda activate smap
pip install earthaccess h5netcdf xarray netCDF4 numpy pandas tqdm ipykernel geopandas rasterio h5py
python -m ipykernel install --user --name smap --display-name "SMAP"

"""

"""
Download SPL4SMAU (SMAP L4 Surface/Rootzone Soil Moisture Analysis Update)
for the Pilbara region, 2015.

Streams files via OPeNDAP using earthaccess so only Pilbara pixels are
pulled — avoids downloading ~2920 full global HDF5 files.

Requirements:
    pip install earthaccess h5netcdf xarray netCDF4 numpy pandas tqdm

NASA Earthdata account required: https://urs.earthdata.nasa.gov
Credentials: data/earthdataLogin.txt, username and password on lines 1 and 2 (git-ignored).
"""

import earthaccess
import xarray as xr
import numpy as np
import pandas as pd
from tqdm import tqdm
import glob
import os
import geopandas as gpd
import h5py
import math
# import dask
from concurrent.futures import ThreadPoolExecutor, as_completed
import rasterio
from rasterio.transform import from_bounds

# ---------------------------------------------------------------------------
# 1. Config
# ---------------------------------------------------------------------------
START_DATE = '2015-03-01' 
# START_DATE = '2024-08-01'
END_DATE = '2024-09-01'

# path to shapefiles for each site
file_location = r"C:\Users\eck025\OneDrive - CSIRO\R directory\PilbaraLaiValidation_clean\data\MultiProjBbox.shp"
shape_file = file_location
site_name = 'AllFortescueSites'
# shape_paths = glob.glob(os.path.join(file_location, "*.shp"))
# names_py = [os.path.splitext(os.path.basename(transect))[0] for transect in shape_paths]
# print(names_py)

# Base of where we want it to go
output_path = r"C:\Users\eck025\OneDrive - CSIRO\R directory\PilbaraLaiValidation_clean\data\spl4smau"

# ---------------------------------------------------------------------------
# 2. Authenticate
# ---------------------------------------------------------------------------
with open(os.path.join(os.path.dirname(file_location), "earthdataLogin.txt")) as f:
    os.environ["EARTHDATA_USERNAME"], os.environ["EARTHDATA_PASSWORD"] = f.read().splitlines()[:2]
earthaccess.login(strategy="environment", persist=True)

C:\Users\eck025\.conda\envs\smap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------------------------------------
# 3. Start of loop - set bounds
# -------------------------------------

# shape_file = shape_paths[0]
# site_name = names_py[0]


# Read bounding box for earthaccess search (must be a box)
roi      = gpd.read_file(shape_file).to_crs("EPSG:4326")
bounds = roi.total_bounds          # [xmin, ymin, xmax, ymax]
LON_MIN = math.floor(bounds[0])
LON_MAX = math.ceil(bounds[2])
LAT_MIN = math.floor(bounds[1])
LAT_MAX = math.ceil(bounds[3])

print(f"Using shapefile: {shape_file}")
print(f"Bounding box: {bounds}")

# HDF5 group and variables to extract
HDF5_GROUP = "Analysis_Data"
VARIABLES  = ["sm_surface_analysis", "sm_rootzone_analysis"]

OUT_DIR  = output_path
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# 4. Search granules
# ---------------------------------------------------------------------------
# results = earthaccess.search_data(
#     short_name   = "SPL4SMAU",
#     temporal     = (START_DATE, END_DATE),
#     bounding_box = (LON_MIN, LAT_MIN, LON_MAX, LAT_MAX),
# )
# print(results)
# print(f"Found {len(results)} granules")   # expect ~2920 per year (8 per day)

Using shapefile: C:\Users\eck025\OneDrive - CSIRO\R directory\PilbaraLaiValidation_clean\data\MultiProjBbox.shp
Bounding box: [113.92591667 -23.31760813 128.77158333 -16.12832615]


In [3]:
# ---------------------------------------------------------------------------
# 4. organise batches
# ---------------------------------------------------------------------------
from datetime import timedelta

TEMP_DIR  = os.path.join("C:/", "temp_large")
BATCH_DAYS = 30
os.makedirs(TEMP_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# generate date windows
start = pd.Timestamp(START_DATE)
end   = pd.Timestamp(END_DATE)
windows = []
current = start
while current < end:
    window_end = min(current + timedelta(days=BATCH_DAYS), end)
    windows.append((current.strftime("%Y-%m-%d"), window_end.strftime("%Y-%m-%d")))
    current = window_end + timedelta(days=1)

print(f"Processing {len(windows)} batches of {BATCH_DAYS} days")


Processing 112 batches of 30 days


In [4]:
# ------------------------------------------------------------------------------------------------
# 5. function to clip the netcdf, select the appropriate variables and re-write as geotiff
# ------------------------------------------------------------------------------------------------
# opens the spatial part of the net cdf file and indexes which cells overlap the polygon
# opens the data bucket and clips it to the right size
# saves it as a geotiff
# import rasterio
# from rasterio.transform import from_bounds
# import rioxarray          # registers .rio accessor on xr objects - allows more 'terra' like functionality with netcdf - attaches crs without being explicit about it
# from pyproj import Transformer

from pyproj import Transformer
import rasterio
from rasterio.transform import from_bounds

_tr = Transformer.from_crs("EPSG:4326", "EPSG:6933", always_xy=True)

def clip_and_save(args):
    temp_file, result = args
    time = result["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
    ts   = pd.Timestamp(time).strftime("%Y%m%dT%H%M%S")
    out  = os.path.join(OUT_DIR, f"spl4smau_{ts}.tif")

    if os.path.exists(out):
        os.remove(temp_file)
        return ts, "skipped"

    try:
        # find row/col indices from geographic coords
        root     = xr.open_dataset(temp_file, engine="h5netcdf", phony_dims="access")
        cell_lat = root["cell_lat"].values
        cell_lon = root["cell_lon"].values
        root.close()

        row_idx = np.where(np.any((cell_lat >= LAT_MIN) & (cell_lat <= LAT_MAX), axis=1))[0]
        col_idx = np.where(np.any((cell_lon >= LON_MIN) & (cell_lon <= LON_MAX), axis=0))[0]

        # get metre corners of the subset
        lat_sub = cell_lat[np.ix_(row_idx, col_idx)]
        lon_sub = cell_lon[np.ix_(row_idx, col_idx)]
        xmin, ymax = _tr.transform(lon_sub[0,   0], lat_sub[0,   0])   # top-left
        xmax, ymin = _tr.transform(lon_sub[-1, -1], lat_sub[-1, -1])   # bottom-right

        # load and subset data
        ds   = xr.open_dataset(temp_file, engine="h5netcdf", group=HDF5_GROUP,
                               mask_and_scale=True, phony_dims="access")
        ds   = ds[VARIABLES].isel(y=row_idx, x=col_idx)
        arrs = [ds[v].values.squeeze() for v in VARIABLES]
        ds.close()

        transform = from_bounds(xmin, ymin, xmax, ymax, arrs[0].shape[1], arrs[0].shape[0])

        with rasterio.open(out, "w",
            driver="GTiff", compress="lzw",
            height=arrs[0].shape[0], width=arrs[0].shape[1],
            count=len(VARIABLES), dtype=arrs[0].dtype,
            crs="EPSG:6933", transform=transform, nodata=-9999,
        ) as dst:
            for i, (arr, var) in enumerate(zip(arrs, VARIABLES), start=1):
                dst.write(arr, i)
                dst.set_band_description(i, var)

        return ts, "saved"

    except Exception as e:
        return ts, f"failed: {e}"

    finally:
        import gc; gc.collect()
        if os.path.exists(temp_file):
            try:
                os.remove(temp_file)
            except PermissionError:
                pass

# ---------------------------------------------------------------------------
# 6. Implement batched workflow
# ---------------------------------------------------------------------------
# searches for data within the batch temporal window
# downloads the full datafiles
# applies function from step 5 to clip the data locally
# save as a geotif
for batch_start, batch_end in tqdm(windows, desc="Batches"):

    results    = earthaccess.search_data(
        short_name   = "SPL4SMAU",
        temporal     = (batch_start, batch_end),
        bounding_box = (LON_MIN, LAT_MIN, LON_MAX, LAT_MAX),
    )
    
    # Filter out granules if the output file already exists
    to_download = []
    for r in results:
        ts = pd.Timestamp(
            r["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
        ).strftime("%Y%m%dT%H%M%S")
        out_path = os.path.join(OUT_DIR, f"spl4smau_{ts}.tif")
        if not os.path.exists(out_path):
            to_download.append(r)

    if not to_download:
        continue  # entire batch already processed

    temp_files = earthaccess.download(to_download, TEMP_DIR)

    # Parallelise the clip step
    with ThreadPoolExecutor(max_workers=os.cpu_count() - 1) as executor:
        futures = [executor.submit(clip_and_save, (f, r))
                   for f, r in zip(temp_files, to_download)]
        for future in as_completed(futures):
            ts, status = future.result()
            if "failed" in status:
                print(f"  {ts}: {status}")

print("Done.")


Batches:   0%|                                                                                 | 0/112 [00:00<?, ?it/s]
QUEUEING TASKS | : 100%|████████████████████████████████████████████████████████████████| 8/8 [00:00<00:00, 886.79it/s]

PROCESSING TASKS | :   0%|                                                                       | 0/8 [00:00<?, ?it/s]
PROCESSING TASKS | :  12%|███████▉                                                       | 1/8 [00:22<02:36, 22.36s/it]
PROCESSING TASKS | :  25%|███████████████▊                                               | 2/8 [00:28<01:15, 12.56s/it]
PROCESSING TASKS | :  38%|███████████████████████▋                                       | 3/8 [00:28<00:34,  6.89s/it]
PROCESSING TASKS | :  50%|███████████████████████████████▌                               | 4/8 [00:28<00:17,  4.32s/it]
PROCESSING TASKS | :  62%|███████████████████████████████████████▍                       | 5/8 [00:29<00:09,  3.11s/it]
PROCESSING TASKS | :  75%|█████████████

Done.


In [ ]:
# ------------------------------------------------------------------------------------------------
# 5. function to clip the netcdf, select the appropriate variables and re-write as geotiff
# ------------------------------------------------------------------------------------------------
# opens the spatial part of the net cdf file and indexes which cells overlap the polygon
# opens the data bucket and clips it to the right size
# saves it as a geotiff
# basically as above but relied on rasterio and low level subsetting, rioxarray hides a bit of that
# also had a problem with setting the CRS of the EASE grids used by smap.


import rasterio
from rasterio.transform import from_bounds

def clip_and_save(args):
    temp_file, result = args
    time = result["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
    ts   = pd.Timestamp(time).strftime("%Y%m%dT%H%M%S")

    if os.path.exists(os.path.join(OUT_DIR, f"spl4smau_{ts}_surface.tif")):
        os.remove(temp_file)
        return ts, "skipped"
    try:
        ds_root  = xr.open_dataset(temp_file, engine="h5netcdf", phony_dims="access")
        cell_lat = ds_root["cell_lat"].values
        cell_lon = ds_root["cell_lon"].values
        ds_root.close()

        row_idx = np.where(np.any((cell_lat >= LAT_MIN) & (cell_lat <= LAT_MAX), axis=1))[0]
        col_idx = np.where(np.any((cell_lon >= LON_MIN) & (cell_lon <= LON_MAX), axis=0))[0]

        lat_vals = cell_lat[np.ix_(row_idx, col_idx)]
        lon_vals = cell_lon[np.ix_(row_idx, col_idx)]

        ds = xr.open_dataset(temp_file, engine="h5netcdf", group=HDF5_GROUP,
                             mask_and_scale=True, phony_dims="access")
        ds = ds[VARIABLES].isel(y=row_idx, x=col_idx)

        transform = from_bounds(
            lon_vals[0, 0], lat_vals[-1, 0],   # xmin, ymin
            lon_vals[0, -1], lat_vals[0, 0],   # xmax, ymax
            len(col_idx), len(row_idx)
        )

        out  = os.path.join(OUT_DIR, f"spl4smau_{ts}.tif")
        arrs = [ds[var].values.squeeze() for var in VARIABLES]
        
        with rasterio.open(
            out, "w",
            driver    = "GTiff",
            height    = arrs[0].shape[0],
            width     = arrs[0].shape[1],
            count     = len(VARIABLES),
            dtype     = arrs[0].dtype,
            crs       = "EPSG:4326",
            transform = transform,
            nodata    = -9999,
        ) as dst:
            for i, (arr, var) in enumerate(zip(arrs, VARIABLES), start=1):
                dst.write(arr, i)
                dst.set_band_description(i, var)
                dst.update_tags(i, variable=var, time=time)

        ds.close()
        del ds
        return ts, "saved"

    except Exception as e:
        return ts, f"failed: {e}"

    finally:
        if os.path.exists(temp_file):
            import gc
            gc.collect()
            os.remove(temp_file)



# ---------------------------------------------------------------------------
# 6. Implement batched workflow
# ---------------------------------------------------------------------------
# searches for data within the batch temporal window
# downloads the full datafiles
# applies function from step 5 to clip the data locally
# save as a geotif
for batch_start, batch_end in tqdm(windows, desc="Batches"):

    results    = earthaccess.search_data(
        short_name   = "SPL4SMAU",
        temporal     = (batch_start, batch_end),
        bounding_box = (LON_MIN, LAT_MIN, LON_MAX, LAT_MAX),
    )
    
    # Filter out granules if the output file already exists
    to_download = []
    for r in results:
        ts = pd.Timestamp(
            r["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
        ).strftime("%Y%m%dT%H%M%S")
        out_path = os.path.join(OUT_DIR, f"spl4smau_{ts}.tif")
        if not os.path.exists(out_path):
            to_download.append(r)

    if not to_download:
        continue  # entire batch already processed

    temp_files = earthaccess.download(to_download, TEMP_DIR)

    # Parallelise the clip step
    with ThreadPoolExecutor(max_workers=os.cpu_count() - 1) as executor:
        futures = [executor.submit(clip_and_save, (f, r))
                   for f, r in zip(temp_files, to_download)]
        for future in as_completed(futures):
            ts, status = future.result()
            if "failed" in status:
                print(f"  {ts}: {status}")

print("Done.")



# for batch_start, batch_end in tqdm(windows, desc="Batches"):

#     results    = earthaccess.search_data(
#         short_name   = "SPL4SMAU",
#         temporal     = (batch_start, batch_end),
#         bounding_box = (LON_MIN, LAT_MIN, LON_MAX, LAT_MAX),
#     )

#     # remove anything that exists already
#     timestamps = results["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
#     timestamps   = pd.Timestamp(timestamps).strftime("%Y%m%dT%H%M%S")
#     output_paths  = os.path.join(OUT_DIR, f"spl4smau_{ts}.tif")
#     # index files that exist
#     results <- results[!idx]
    
#     temp_files = earthaccess.download(results, TEMP_DIR)

#     # parallelise the clip step
#     with ThreadPoolExecutor(max_workers=os.cpu_count()-1) as executor:
#         futures = [executor.submit(clip_and_save, (f, r)) 
#                    for f, r in zip(temp_files, results)]
#         for future in as_completed(futures):
#             ts, status = future.result()
#             if "failed" in status:
#                 print(f"  {ts}: {status}")

# print("Done.")

In [ ]:
###################################
# Debugging....
###################################

batch_start, batch_end = windows[0]
results    = earthaccess.search_data(
    short_name   = "SPL4SMAU",
    temporal     = (batch_start, batch_end),
    bounding_box = (LON_MIN, LAT_MIN, LON_MAX, LAT_MAX),
)
temp_files = earthaccess.download(results, TEMP_DIR)

args = (temp_files[0],results[0])
temp_file, result = args
time = result["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
ts   = pd.Timestamp(time).strftime("%Y%m%dT%H%M%S")

ds_root  = xr.open_dataset(temp_file, engine="h5netcdf", phony_dims="access")

cell_lat = ds_root["cell_lat"].values
cell_lon = ds_root["cell_lon"].values


row_idx = np.where(np.any((cell_lat >= LAT_MIN) & (cell_lat <= LAT_MAX), axis=1))[0]
col_idx = np.where(np.any((cell_lon >= LON_MIN) & (cell_lon <= LON_MAX), axis=0))[0]

print(row_idx)


lat_vals = cell_lat[np.ix_(row_idx, col_idx)]
lon_vals = cell_lon[np.ix_(row_idx, col_idx)]

ds = xr.open_dataset(temp_file, engine="h5netcdf", group=HDF5_GROUP,
                     mask_and_scale=True, phony_dims="access")
ds = ds[VARIABLES].isel(y=row_idx, x=col_idx)

transform = from_bounds(
    lon_vals[0, 0], lat_vals[-1, 0],
    lon_vals[0, -1], lat_vals[0, 0],
    len(col_idx), len(row_idx)
)

print(transform)
out  = os.path.join(OUT_DIR, f"spl4smau_{ts}.tif")
arrs = [ds[var].values.squeeze() for var in VARIABLES]

with rasterio.open(
    out, "w",
    driver    = "GTiff",
    height    = arrs[0].shape[0],
    width     = arrs[0].shape[1],
    count     = len(VARIABLES),
    dtype     = arrs[0].dtype,
    crs       = "EPSG:4326",
    transform = transform,
    nodata    = -9999,
) as dst:
    for i, (arr, var) in enumerate(zip(arrs, VARIABLES), start=1):
        dst.write(arr, i)
        dst.update_tags(i, variable=var, time=time)

ds.close()

In [ ]:
def clip_and_save(args):
    temp_file, result = args
    time = result["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
    ts   = pd.Timestamp(time).strftime("%Y%m%dT%H%M%S")
    out  = os.path.join(OUT_DIR, f"spl4smau_{ts}.nc")
    if os.path.exists(out):
        os.remove(temp_file)
        return ts, "skipped"
    try:
        # get 2D lat/lon from root group and find row/col indices
        ds_root  = xr.open_dataset(temp_file, engine="h5netcdf", phony_dims="access")
        cell_lat = ds_root["cell_lat"].values
        cell_lon = ds_root["cell_lon"].values
        ds_root.close()

        row_mask = np.any((cell_lat >= LAT_MIN) & (cell_lat <= LAT_MAX), axis=1)
        col_mask = np.any((cell_lon >= LON_MIN) & (cell_lon <= LON_MAX), axis=0)
        row_idx  = np.where(row_mask)[0]
        col_idx  = np.where(col_mask)[0]

        # open analysis group and subset by index
        ds = xr.open_dataset(
            temp_file,
            engine         = "h5netcdf",
            group          = HDF5_GROUP,
            mask_and_scale = True,
            phony_dims     = "access"
        )
        ds = ds[VARIABLES]
        ds = ds.isel(y=row_idx, x=col_idx)
        ds = ds.expand_dims(time=[time])

        # assign 1D lat/lon coords (mean per row/col — valid for small regions)
        lat_vals = cell_lat[np.ix_(row_idx, col_idx)]
        lon_vals = cell_lon[np.ix_(row_idx, col_idx)]
        ds = ds.assign_coords(
            y=("y", lat_vals[:, 0]),
            x=("x", lon_vals[0, :]),
        )
        ds = ds.rename({"y": "lat", "x": "lon"})

        ds.to_netcdf(out)
        ds.close()
        del ds
        return ts, "saved"

    except Exception as e:
        return ts, f"failed: {e}"

    finally:
        if os.path.exists(temp_file):
            import gc
            gc.collect()
            os.remove(temp_file)

In [ ]:
temp_file = temp_files[0]
time = results[0]["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]

clip_and_save((temp_file,results[0]))

# ds_root = xr.open_dataset(
#     temp_file,
#     engine     = "h5netcdf",
#     phony_dims = "access",
# )

# # cell_lat/lon are 2D (y, x) — find row/col indices within bounds
# cell_lat = ds_root["cell_lat"].values   # shape (1624, 3856)
# cell_lon = ds_root["cell_lon"].values

# # find indices where both lat and lon fall within bounds
# row_mask = np.any((cell_lat >= LAT_MIN) & (cell_lat <= LAT_MAX), axis=1)
# col_mask = np.any((cell_lon >= LON_MIN) & (cell_lon <= LON_MAX), axis=0)

# row_idx  = np.where(row_mask)[0]
# col_idx  = np.where(col_mask)[0]

# # extract matching lat/lon values for coordinate assignment
# lat_vals = cell_lat[np.ix_(row_idx, col_idx)]
# lon_vals = cell_lon[np.ix_(row_idx, col_idx)]
# ds_root.close()

# # load and subset Analysis_Data using integer indices
# ds = xr.open_dataset(
#     temp_file,
#     engine         = "h5netcdf",
#     group          = HDF5_GROUP,
#     mask_and_scale = True,
#     phony_dims     = "access",
# )
# ds = ds[VARIABLES]
# ds = ds.isel(y=row_idx, x=col_idx)
# ds = ds.expand_dims(time=[time])

# # assign 2D lat/lon as coordinates
# ds = ds.assign_coords(
#     lat=(["y", "x"], lat_vals),
#     lon=(["y", "x"], lon_vals),
# )

# print(ds)
# # get actual lat/lon coordinate arrays from root group
# # cell_lon = ds_root["cell_lon"].values   # confirm name from print(ds_root)
# # cell_lat = ds_root["cell_lat"].values
# # ds_root.close()

In [ ]:
def clip_and_save(args):
    temp_file, result = args
    time = result["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
    ts   = pd.Timestamp(time).strftime("%Y%m%dT%H%M%S")
    out  = os.path.join(OUT_DIR, f"spl4smau_{ts}.nc")

    if os.path.exists(out):
        os.remove(temp_file)
        return ts, "skipped"

    try:
        ds = xr.open_dataset(
            temp_file,
            engine = "h5netcdf",
            group = HDF5_GROUP,
            mask_and_scale = True,
            phony_dims = "access"
        )
        ds = ds[VARIABLES]
        ds = ds.sel(x=slice(LON_MIN, LON_MAX), y=slice(LAT_MIN, LAT_MAX))
        ds = ds.expand_dims(time=[time])
        ds.to_netcdf(out)
        ds.close()
        del ds          # explicitly release handle before deletion
        return ts, "saved"
        
    except Exception as e:
        return ts, f"failed: {e}"
        
    finally:
        if os.path.exists(temp_file):
            import gc
            gc.collect()        # force garbage collection to release any lingering handles
            os.remove(temp_file)

for batch_start, batch_end in tqdm(windows, desc="Batches"):

    results    = earthaccess.search_data(
        short_name   = "SPL4SMAU",
        temporal     = (batch_start, batch_end),
        bounding_box = (LON_MIN, LAT_MIN, LON_MAX, LAT_MAX),
    )
    temp_files = earthaccess.download(results, TEMP_DIR)

    # parallelise the clip step
    with ThreadPoolExecutor(max_workers=os.cpu_count()-1) as executor:
        futures = [executor.submit(clip_and_save, (f, r)) 
                   for f, r in zip(temp_files, results)]
        for future in as_completed(futures):
            ts, status = future.result()
            if "failed" in status:
                print(f"  {ts}: {status}")

print("Done.")

In [ ]:
#### use to examine single cells
# files = earthaccess.open(results)
# file = files[0]
# ds = xr.open_dataset(
#     file,
#     group          = HDF5_GROUP,
#     mask_and_scale = True,
#     decode_times   = True,
#     phony_dims     = "access",
# )
# ds = ds.sel(x=slice(LON_MIN, LON_MAX), y=slice(LAT_MIN, LAT_MAX))
# time = results[0]["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
# ds = ds.expand_dims(time=[time])
# pilbara = ds.sel(x=slice(LON_MIN,LON_MAX), y=slice(LAT_MIN,LAT_MAX))

# print(pilbara)

In [ ]:
# alternative to above. it worked but was a bit shit.
from concurrent.futures import ThreadPoolExecutor, as_completed

def open_one(args):
    i, file, result = args
    
    time = result["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
    ts   = pd.Timestamp(time).strftime("%Y%m%dT%H%M%S")
    out  = os.path.join(OUT_DIR, f"spl4smau_{ts}.nc")
    
    # skip if already downloaded
    if os.path.exists(out):
        return i, ts, "skipped"
    
    ds = xr.open_dataset(
        file,
        group          = HDF5_GROUP,
        mask_and_scale = True,
        decode_times   = True,
        phony_dims     = "access",
    )
    ds = ds.sel(x=slice(LON_MIN, LON_MAX), y=slice(LAT_MIN, LAT_MAX))
    ds = ds.expand_dims(time=[time])
    ds = ds[['sm_surface_analysis', 'sm_rootzone_analysis']]
    ds.to_netcdf(out)
    return i, ts, "saved"

files = earthaccess.open(results)

MAX_WORKERS = min(20, len(files))
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(open_one, (i, f, r))
               for i, (f, r) in enumerate(zip(files, results))]

    with tqdm(total=len(futures), desc="Downloading granules") as pbar:
        for future in as_completed(futures):
            try:
                i, ts, status = future.result()
                pbar.set_postfix({"last": ts, "status": status})
            except Exception as e:
                pbar.write(f"  Failed: {e}")
            pbar.update(1)


In [ ]:
files = earthaccess.open(results)

def preprocess(ds):
    ds = ds.sel(
        x=slice(LON_MIN, LON_MAX),
        y=slice(LAT_MIN, LAT_MAX)
    )
    return ds

ds_all = xr.open_mfdataset(
    files,
    engine="h5netcdf",
    group=HDF5_GROUP,
    mask_and_scale=True,
    decode_times=True,
    phony_dims="access",
    preprocess=preprocess,
    concat_dim="time",
    combine="nested",
    chunks="auto"
)

# this triggers computation & downloads everything
ds_all.to_netcdf(OUT_FILE)
    

In [ ]:
# this triggers computation & downloads everything
ds_all.to_netcdf(OUT_FILE)

In [ ]:
# -------------------------------------
# 3. Start of loop - set bounds
# -------------------------------------

shape_file = shape_paths[0]
site_name = names_py[0]


# Read bounding box for earthaccess search (must be a box)
roi      = gpd.read_file(shape_file).to_crs("EPSG:4326")
bounds               = roi.total_bounds          # [xmin, ymin, xmax, ymax]
LON_MIN, LON_MAX     = bounds[0], bounds[2]
LAT_MIN, LAT_MAX     = bounds[1], bounds[3]

print(f"Using shapefile: {shape_file}")
print(f"Bounding box: {bounds}")

# HDF5 group and variables to extract
HDF5_GROUP = "Analysis_Data"
VARIABLES  = ["sm_surface_analysis", "sm_rootzone_analysis"]

OUT_DIR  = os.path.join(output_path, site_name)
FILE_NAME = "spl4smau_"+START_DATE+"_"+END_DATE+".nc"
OUT_FILE = os.path.join(OUT_DIR,FILE_NAME)

os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# 4. Search granules
# ---------------------------------------------------------------------------
results = earthaccess.search_data(
    short_name   = "SPL4SMAU",
    temporal     = (START_DATE, END_DATE),
    bounding_box = (LON_MIN, LAT_MIN, LON_MAX, LAT_MAX),
)
print(f"Found {len(results)} granules")   # expect ~2920 per year (8 per day)


# ---------------------------------------------------------------------------
# 5. Stream and spatially subset via OPeNDAP
#    earthaccess.open() returns file-like objects; xarray reads via h5netcdf
#    so only the subsetted data is transferred.
# ---------------------------------------------------------------------------


def open_and_subset(file_obj):
    PIXEL_DEG = 0.20   # ~18 km in degrees — buffer applied to all polygons

    ds_full = xr.open_dataset(
        file_obj,
        group          = HDF5_GROUP,
        engine         = "h5netcdf",
        mask_and_scale = True,
        phony_dims     = "access",
    )

    # inspect actual coord names on first run, then hardcode
    print(ds_full.dims, ds_full.coords)

    lons = ds_full["x"].values    # update "x" once confirmed from print above
    lats = ds_full["y"].values    # update "y" once confirmed from print above

    lon_mask = (lons >= LON_MIN - PIXEL_DEG) & (lons <= LON_MAX + PIXEL_DEG)
    lat_mask = (lats >= LAT_MIN - PIXEL_DEG) & (lats <= LAT_MAX + PIXEL_DEG)

    subset_vars = {}
    for var in VARIABLES:
        if var in ds_full:
            arr = ds_full[var].values
            subset_vars[var] = (["y", "x"], arr[np.ix_(lat_mask, lon_mask)])

    ds_sub = xr.Dataset(
        subset_vars,
        coords={"x": lons[lon_mask], "y": lats[lat_mask]},
    )
    ds_full.close()
    return ds_sub


# ---------------------------------------------------------------------------
# 6. Parse timestamps from granule metadata and iterate
# ---------------------------------------------------------------------------
datasets   = []
timestamps = []

file_objects = earthaccess.open(results)

for i, (result, fobj) in enumerate(tqdm(
        zip(results, file_objects), total=len(results),
        desc="Streaming granules")):

    # Extract timestamp from granule UR  e.g. "...20150101T013000..."
    ur = result["meta"]["native-id"]
    try:
        # SPL4SMAU granule names contain datetime like 20150101T013000
        import re
        ts_str = re.search(r"(\d{8}T\d{6})", ur).group(1)
        ts = pd.Timestamp(ts_str)
    except Exception:
        ts = pd.Timestamp(START_DATE) + pd.Timedelta(hours=3 * i)

    try:
        ds_sub = open_and_subset(fobj)
        ds_sub = ds_sub.expand_dims(time=[ts])
        datasets.append(ds_sub)
    except Exception as e:
        print(f"  Warning: skipped granule {i} ({ts}): {e}")

# ---------------------------------------------------------------------------
# 7. Concatenate and save
# ---------------------------------------------------------------------------
print("Concatenating...")
combined = xr.concat(datasets, dim="time")

# Attach CRS info as attributes
combined.attrs.update({
    "crs"         : "EPSG:4326",
    "product"     : "SPL4SMAU",
    "region"      : "Pilbara, Western Australia",
    "lon_bounds"  : f"{LON_MIN}, {LON_MAX}",
    "lat_bounds"  : f"{LAT_MIN}, {LAT_MAX}",
    "source"      : "NASA NSIDC DAAC",
})
combined["sm_surface_analysis"].attrs["units"]  = "m³/m³"
combined["sm_rootzone_analysis"].attrs["units"] = "m³/m³"

combined.to_netcdf(OUT_FILE)
print(f"Saved → {OUT_FILE}")
print(combined)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---------------------------------------------------------------------------
# Define function for one site
# ---------------------------------------------------------------------------
def process_site(shape_file, site_name,start_date,end_date):

    print(f"[{site_name}] Starting...")

    ########################
    # 1. Define bounds
    ########################
    roi      = gpd.read_file(shape_file).to_crs("EPSG:4326")
    bounds   = roi.total_bounds
    LON_MIN, LON_MAX = bounds[0], bounds[2]
    LAT_MIN, LAT_MAX = bounds[1], bounds[3]

    HDF5_GROUP = "Analysis_Data"
    VARIABLES  = ["sm_surface_analysis", "sm_rootzone_analysis"]

    OUT_DIR  = os.path.join(output_path, site_name)
    FILE_NAME = "spl4smau_" + start_date + "_" + end_date + ".nc"
    OUT_FILE  = os.path.join(OUT_DIR, FILE_NAME)
    os.makedirs(OUT_DIR, exist_ok=True)

    ################################
    # Search granules on server
    ################################
    results = earthaccess.search_data(
        short_name   = "SPL4SMAU",
        temporal     = (start_date, end_date),
        bounding_box = (LON_MIN, LAT_MIN, LON_MAX, LAT_MAX),
    )
    print(f"[{site_name}] Found {len(results)} granules")

    ################################
    # subset the ones I want
    ################################
    # Open one SPL4SMAU HDF5, subset to Pilbara, return small xr.Dataset
    def open_and_subset(file_obj):
        ds_full = xr.open_dataset(
            file_obj,
            group          = HDF5_GROUP,
            engine         = "h5netcdf",
            mask_and_scale = True,
        )

        # Coordinates are named 'x' (lon) and 'y' (lat) in the EASE-Grid 2.0 file
        # Rename if necessary so spatial selection works cleanly
        if "x" in ds_full.dims and "y" in ds_full.dims:
            ds_full = ds_full.rename({"x": "lon", "y": "lat"})
            
        # Load coordinate values for masking (small — just 1D arrays)
        lons = ds_full["lon"].values if "lon" in ds_full.coords else None
        lats = ds_full["lat"].values if "lat" in ds_full.coords else None
        
        # Boolean masks
        lon_mask = (lons >= LON_MIN) & (lons <= LON_MAX)
        lat_mask = (lats >= LAT_MIN) & (lats <= LAT_MAX)
    
        # Subset variables
        subset_vars = {}
        for var in VARIABLES:
            if var in ds_full:
                arr = ds_full[var].values  # 2-D (lat x lon)
                arr_sub = arr[np.ix_(lat_mask, lon_mask)]
                subset_vars[var] = (["lat", "lon"], arr_sub)
    
        ds_sub = xr.Dataset(
            subset_vars,
            coords={
                "lon": lons[lon_mask],
                "lat": lats[lat_mask],
            },
        )
        ds_full.close()
        return ds_sub

    # Stream and subset
    datasets     = []
    file_objects = earthaccess.open(results)

    for i, (result, fobj) in enumerate(zip(results, file_objects)):
        ur = result["meta"]["native-id"]
        try:
            ts_str = re.search(r"(\d{8}T\d{6})", ur).group(1)
            ts = pd.Timestamp(ts_str)
        except Exception:
            ts = pd.Timestamp(start_date) + pd.Timedelta(hours=3 * i)
        try:
            ds_sub = open_and_subset(fobj)
            ds_sub = ds_sub.expand_dims(time=[ts])
            datasets.append(ds_sub)
        except Exception as e:
            print(f"  [{site_name}] Warning: skipped granule {i} ({ts}): {e}")

    # Concatenate and save
    combined = xr.concat(datasets, dim="time")
    combined.attrs.update({
        "crs": "EPSG:4326", "product": "SPL4SMAU", "source": "NASA NSIDC DAAC",
        "lon_bounds": f"{LON_MIN}, {LON_MAX}", "lat_bounds": f"{LAT_MIN}, {LAT_MAX}",
    })
    combined["sm_surface_analysis"].attrs["units"]  = "m³/m³"
    combined["sm_rootzone_analysis"].attrs["units"] = "m³/m³"
    combined.to_netcdf(OUT_FILE)
    print(f"[{site_name}] Saved → {OUT_FILE}")
    return site_name